# Native randomized sequence and inverse experiment

Retain the original one-qubit size, five independently generated Clifford words
of lengths 5, 10, 15, 20 and 25, Python RNG seed 0, hardware-list filtering order
`id`, `rz`, `sx`, `x`, RZ(pi), and full word followed by its checked adjoint.
SX includes its absolute scalar phase; it is not replaced by RX alone.
Five models are synthesized concurrently. Ten execution jobs run in batches of
at most three, with each job retaining its actual model length and mode.

The source uses two hosted simulator names without a specified shot count. This
local counterpart explicitly uses two actual FlagQuantum numerical modes,
statevector and density matrix, with exact CPU complex128 execution and 1024
seeded samples per job. These are local representation checks, not access to
Classiq services or two hardware devices. The public synthesized `program.circuit`
is executed with explicit options; the checked `fq.run(program)` path is also
validated separately. No sealed Program execution override is attempted.

The source labels its ten product-ordered jobs with a repeated five-element
length list. Eight jobs receive the wrong length. We show that mismatch and
plot each result using its own executed job metadata. Ideal sequence/inverse
observations cannot identify a noisy-device randomized-benchmarking decay or
error per Clifford. No fitted hardware error, sampled historical result,
speedup, cloud execution or scalability claim is made.


In [ ]:
from pathlib import Path

from IPython.display import Image, display

from examples.synthesis.applications.benchmarking.randomized_benchmarking.overview.ansatz import (
    fresh_sequences,
    make_model,
)
from examples.synthesis.applications.benchmarking.randomized_benchmarking.overview.plots import (
    plot_experiment,
)
from examples.synthesis.applications.benchmarking.randomized_benchmarking.overview.workflow import (
    Experiment,
    execute_batches,
    source_label_mismatches,
    synthesize_all,
)

sequences = fresh_sequences(0)
models = tuple(make_model(sequence) for sequence in sequences)
print("Original lengths:", tuple(map(len, sequences)))
print("Fresh native model count:", len(models))
print("Fresh seeded words:", sequences)


In [ ]:
programs = await synthesize_all(models)
print("Synthesized five original-length programs concurrently:", len(programs))
print(
    "Actual native resources:",
    tuple(program.report["resources"] for program in programs),
)


In [ ]:
trials, batch_sizes = await execute_batches(programs, sequences, shots=1024)
print("Actual bounded execution batches:", batch_sizes)
for row in trials:
    print(row)
print("Total actual samples:", sum(sum(row.counts) for row in trials))


In [ ]:
mismatches = source_label_mismatches(trials)
experiment = Experiment(sequences, programs, trials, batch_sizes, mismatches)
print("Source repeated-length labeling mismatches:", mismatches)
print("Analysis groups are keyed by each executed job's actual length and local mode")
print(
    "All ideal return probabilities equal one; noisy-device decay is not identifiable"
)
figure = plot_experiment(
    experiment, Path("artifacts/development/randomized_benchmarking")
)
display(Image(filename=str(figure)))
